# Package installation

In [ ]:
#%pip install numpy
#%pip install pillow
#%pip install matplotlib
#%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
from PIL import Image
from itertools import permutations
from typing import List
from itertools import permutations
import matplotlib.pyplot as plt
import tifffile
import secrets

# ALgorith preparation

## Matrix creation

In [ ]:
def generar_matrices_4x4(seed):
    if seed is not None:
        np.random.seed(seed)

    conjuntos = []

    rangos = [
        (0, 24),  # 0 - 24
        (24, 48),  # 39 - 69
        (48, 72),  # otro rango distinto (puedes ajustarlo)
    ]

    for low, high in rangos:
        ops = [
            np.random.randint(low, high, size=(2, 2)).astype(float) for _ in range(4)
        ]

        matrices_4x4 = [
            np.block([[M1, M2], [M3, M4]]) for M1, M2, M3, M4 in permutations(ops, 4)
        ]

        conjuntos.append(matrices_4x4)

    print("RED", conjuntos[0])
    print("BLUE", conjuntos[1])
    print("GREEN", conjuntos[2])
    return conjuntos


def matrices_encriptacion(llave: List[int], matrices_rotacion):
    matrices_encript = []
    for item in llave:
        matrices_encript.append(matrices_rotacion[item % 24])
    return matrices_encript

## KEYS

In [ ]:
def generar_llave(n: int) -> List[List[int]]:
    """
    Genera una llave de tamaño n (n % 3 == 0),
    aplica DTQW global, luego divide en 3 partes iguales
    y aplica DTQW nuevamente a cada segmento.
    Retorna un array de longitud 3.
    """

    if n < 15 or n % 3 != 0:
        raise ValueError("n debe ser >= 15 y divisible por 3")

    seed = np.random.SeedSequence().entropy
    rng = np.random.default_rng(seed)

    def generar_aleatoria():
        return rng.integers(0, 100, size=n).tolist()

    def coin_matrix(theta):
        return np.array(
            [[np.cos(theta), np.sin(theta)], [np.sin(theta), -np.cos(theta)]]
        )

    def shift(psi):
        psi_new = np.zeros_like(psi)
        psi_new[0, :] = np.roll(psi[0, :], -1)
        psi_new[1, :] = np.roll(psi[1, :], 1)
        return psi_new

    def reordenar(K):
        m = len(K)

        suma = K[0] + K[1] + K[2]
        theta1 = (K[0] / suma) * (np.pi / 2)
        theta2 = (K[1] / suma) * (np.pi / 2)
        theta3 = (K[2] / suma) * (np.pi / 2)

        m_bits = [int(b) for b in format(K[3], "08b")]
        r_pasos = K[3] + K[4]

        psi = np.zeros((2, m), dtype=complex)
        psi[0, 0] = 1.0

        for t in range(r_pasos):
            if t < len(m_bits):
                theta = theta1 if m_bits[t] == 0 else theta2
            else:
                theta = theta3

            C = coin_matrix(theta)
            psi_after = np.zeros_like(psi)

            for j in range(m):
                psi_after[:, j] = C @ psi[:, j]

            psi = shift(psi_after)

        probs = np.abs(psi[0]) ** 2 + np.abs(psi[1]) ** 2
        orden = np.argsort(probs)[::-1]

        return [K[i] for i in orden]

    # 1. Generar llave base
    K = generar_aleatoria()

    # 2. DTQW global
    K_global = reordenar(K)

    # 3. Dividir en 3 partes
    size = n // 3
    K1 = K_global[:size]
    K2 = K_global[size : 2 * size]
    K3 = K_global[2 * size :]

    # 4. DTQW por segmento
    K1 = reordenar(K1)
    K2 = reordenar(K2)
    K3 = reordenar(K3)

    # 5. Retorno
    return [K1, K2, K3]

## Image

In [ ]:
# Funcion meramente cosmetica, basicamente para poder ver las imagenes con sus colores una vez obtenemos el RGB
def visualizar_canal_color(canal_img: Image.Image, color: str) -> Image.Image:
    color = color.upper()
    if color not in ["R", "G", "B"]:
        raise ValueError()

    arr = np.array(canal_img)
    zeros = np.zeros_like(arr)

    if color == "R":
        rgb = np.stack([arr, zeros, zeros], axis=2)
    elif color == "G":
        rgb = np.stack([zeros, arr, zeros], axis=2)
    else:
        rgb = np.stack([zeros, zeros, arr], axis=2)

    return Image.fromarray(rgb, mode="RGB")


def obtener_rgb(ruta_png: str):
    img = Image.open(ruta_png).convert("RGB")
    r_img, g_img, b_img = img.split()

    r = r_img  # (h, w)
    g = g_img  # (h, w)
    b = b_img  # (h, w)

    return r, g, b

## Cipher

In [ ]:
def obtener_rgb_como_bloques(R, G, B):
    """
    Convierte cada canal a bloques (n, 4) mediante flatten + reshape,
    sin interpolación, para que el cifrado matricial sea reversible.
    """

    def canal_a_bloques(canal_img):
        arr = np.array(canal_img).flatten().astype(float)
        pad = (4 - len(arr) % 4) % 4
        if pad:
            arr = np.concatenate([arr, np.zeros(pad)])
        return arr.reshape(4, -1)

    return canal_a_bloques(R), canal_a_bloques(G), canal_a_bloques(B)


def canal_a_bloques(canal_arr: np.ndarray) -> np.ndarray:
    """
    Versión standalone de canal_a_bloques para operar sobre arrays 2D ya permutados.
    Recibe un np.ndarray (h, w), retorna (N, 4).
    """
    arr = canal_arr.flatten().astype(float)
    pad = (4 - len(arr) % 4) % 4
    if pad:
        arr = np.concatenate([arr, np.zeros(pad)])
    return arr.reshape(4, -1)


def cifrar_canal(canal: np.ndarray, matrices_finales: list[np.ndarray]) -> np.ndarray:
    resultado = canal.astype(float)
    for Mi in matrices_finales:
        resultado = Mi @ resultado
    return resultado


def generar_matriz_mezcla(seed_val: int) -> np.ndarray:
    rng = np.random.default_rng(seed_val)
    A = rng.standard_normal((3, 3))
    Q, R_qr = np.linalg.qr(A)
    signs = np.sign(np.diag(R_qr))
    return Q * signs


def mezclar_canales(
    R_b: np.ndarray, G_b: np.ndarray, B_b: np.ndarray, M_mix: np.ndarray
):
    stack = np.stack([R_b, G_b, B_b], axis=0)  # (3, 4, N)
    stack = stack.reshape(3, -1)  # (3, 4N)
    mezclado = M_mix @ stack
    mezclado = mezclado.reshape(3, 4, -1)
    return (mezclado[0], mezclado[1], mezclado[2])


def permutar_filas_columnas(canal_2d: np.ndarray, seed_val: int):
    """
    Permutación independiente de filas y columnas sobre el canal 2D.
    Retorna el canal permutado y los índices para invertir.
    seed_val+2 para filas, seed_val+3 para columnas.
    """
    rng_f = np.random.default_rng(seed_val + 2)
    rng_c = np.random.default_rng(seed_val + 3)
    idx_filas = rng_f.permutation(canal_2d.shape[0])
    idx_cols = rng_c.permutation(canal_2d.shape[1])
    return canal_2d[idx_filas, :][:, idx_cols]


def permutar_bloques(R_b: np.ndarray, G_b: np.ndarray, B_b: np.ndarray, seed_val: int):

    N = R_b.shape[1]

    rng = np.random.default_rng(seed_val + 1)
    idx_r = rng.permutation(N)
    idx_g = rng.permutation(N)
    idx_b = rng.permutation(N)

    return (R_b[:, idx_r], G_b[:, idx_g], B_b[:, idx_b])


def reescalar_imagen_cifrada(c_r_2d, c_g_2d, c_b_2d):
    def normalizar(canal):
        mn, mx = canal.min(), canal.max()
        return (canal - mn) / (mx - mn) * 255.0 if mx != mn else np.zeros_like(canal)

    return normalizar(c_r_2d), normalizar(c_g_2d), normalizar(c_b_2d)


def construir_llave_permutacion(keys: list) -> np.ndarray:
    K1, K2, K3 = np.array(keys[0]), np.array(keys[1]), np.array(keys[2])
    return K1 + K2 + K3


def rotar_columnas(imagen: np.ndarray, K: np.ndarray) -> np.ndarray:
    resultado = imagen.copy()
    h_img, w_img = imagen.shape
    n = len(K)
    for i in range(w_img):
        desplazamiento = int(K[i % n]) % h_img
        resultado[:, i] = np.roll(imagen[:, i], desplazamiento)
    return resultado


def permutacion_filas_columnas(imagen: np.ndarray, K: np.ndarray) -> np.ndarray:
    img_col = rotar_columnas(imagen, K)
    img_fila = rotar_columnas(img_col.T, K).T
    return img_fila

# Cipher Implementation Algorithm

In [ ]:
path = r"../Imagenes/4.1.08.tiff"
R, G, B = obtener_rgb(path)
h, w = R.height, R.width

# 1. Cargar canales como arrays 2D
R_arr = np.array(R)  # (h, w)
G_arr = np.array(G)
B_arr = np.array(B)

seed = secrets.randbits(32)

# 2. Generar estructuras
keys = generar_llave(327)
matrices_sets = generar_matrices_4x4(seed)

# ── ETAPA A: Permutación 2D (filas + columnas) — rompe correlación H y V ───
R_arr_p = permutar_filas_columnas(R_arr, seed)
G_arr_p = permutar_filas_columnas(G_arr, seed)
B_arr_p = permutar_filas_columnas(B_arr, seed)

# ── ETAPA B: Convertir a bloques ───────────────────────────────────────────
R_b = canal_a_bloques(R_arr_p)
G_b = canal_a_bloques(G_arr_p)
B_b = canal_a_bloques(B_arr_p)

# ── ETAPA C: Mezcla inter-canal ────────────────────────────────────────────
M_mix = generar_matriz_mezcla(seed)
R_m, G_m, B_m = mezclar_canales(R_b, G_b, B_b, M_mix)

# 3. Encriptaciones por canal
encriptacion_R = matrices_encriptacion(keys[0], matrices_sets[0])
encriptacion_G = matrices_encriptacion(keys[1], matrices_sets[1])
encriptacion_B = matrices_encriptacion(keys[2], matrices_sets[2])

# 4. Cifrado matricial
c_r = cifrar_canal(R_m, encriptacion_R)
c_g = cifrar_canal(G_m, encriptacion_G)
c_b = cifrar_canal(B_m, encriptacion_B)

# 5. Reconstrucción 2D
c_r_2d = c_r.reshape(-1)[: h * w].reshape(h, w)
c_g_2d = c_g.reshape(-1)[: h * w].reshape(h, w)
c_b_2d = c_b.reshape(-1)[: h * w].reshape(h, w)

# ── ETAPA E: Permutación post-cifrado (sin reescalar) ─────────────────────
K_perm = construir_llave_permutacion(keys)

r_perm = permutacion_filas_columnas(c_r_2d, K_perm)
g_perm = permutacion_filas_columnas(c_g_2d, K_perm)
b_perm = permutacion_filas_columnas(c_b_2d, K_perm)

# 6. Guardado como float64 nativo (TIFF soporta rangos arbitrarios)
cifrado_stack = np.stack(
    [r_perm.astype(np.float64), g_perm.astype(np.float64), b_perm.astype(np.float64)],
    axis=0,
)

cifrado_hwc = np.transpose(cifrado_stack, (1, 2, 0))
tifffile.imwrite(r"../Imagenes/cifrado.tiff", cifrado_hwc.astype(np.float64))

tiffile_leido = tifffile.imread(r"../Imagenes/cifrado.tiff")
print(tiffile_leido.shape)
print(tiffile_leido[:3, :4, 0])
print(tiffile_leido[:3, :4, 1])
print(tiffile_leido[:3, :4, 2])